# 15d. Hybrid GAT + LightGBM dạng stacking, xác suất lớp out-of-fold (task 5.7, GĐ F)

Thay embedding 64 chiều (thất bại ở notebook 15, 15b vì mỗi GAT nhóm có không gian embedding riêng) bằng xác suất 5 lớp của GAT, có ngữ nghĩa cố định giữa các GAT. Đặc trưng LightGBM = 18 đặc trưng node + K xác suất. Train: xác suất out-of-fold (nhóm f từ GAT nhóm f); val: trung bình xác suất k GAT nhóm.

**Không huấn luyện GAT lại**: nạp checkpoint `gat_fold<f>.pt` của notebook 15b (huấn luyện chỉ trên train). **CPU, 0 giờ GPU.** Ghi `model=hybrid_gat_lgbm_stack`, 3 seed × 2 bộ, chỉ `split=val`. **Không đọc test.**

Gắn: output của 04, 11 (đồ thị), 15b (`models_hybrid/*/gat_fold*.pt`, `hybrid_search_oofavg_*.json`), 15 (`models_hybrid/*/gat.pt`, chỉ để chẩn đoán). Bật Internet.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
INFO["device"] = "GPU" if torch.cuda.is_available() else "CPU"
INFO["cpu_count"] = os.cpu_count()
print("Commit:", INFO["commit"], "| máy:", INFO["device"])
assert INFO["device"] == "CPU", "Notebook này chạy CPU, không bật GPU"

## 1. pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Stacking, mỗi bộ, 3 seed

In [ ]:
DATASETS = ["nf_bot_iot_v3", "nf_ton_iot_v3"]
MODELS = f"{WORK}/models_hybrid"


def root_of(kernel, pattern):
    """Thư mục `models_hybrid` của một notebook đã gắn, tìm theo một file mẫu."""
    hit = next(glob.iglob(f"/kaggle/input/**/{kernel}/models_hybrid/{pattern}", recursive=True))
    return str(Path(hit).parents[1])


for ds in DATASETS:
    data_dir = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    folds_dir = root_of("gnn-iot-ids-15b-hybrid-oofavg", f"{ds}_s42/gat_fold0.pt")
    full_dir = root_of("gnn-iot-ids-15-hybrid", f"{ds}_s42/gat.pt")
    fold_search = next(glob.iglob(f"/kaggle/input/**/gnn-iot-ids-15b-hybrid-oofavg/hybrid_search_oofavg_{ds}.json", recursive=True))
    res, search = f"{WORK}/results_hybrid_stack_{ds}.csv", f"{WORK}/hybrid_search_stack_{ds}.json"
    for p in (res, search):
        assert not os.path.exists(p), f"{p} đã có từ trước"
    print("=====", ds, "\n  dữ liệu:", data_dir, "\n  GAT nhóm:", folds_dir, "\n  GAT toàn train:", full_dir)
    with step(f"stack_{ds}"):
        run([sys.executable, "-I", "scripts/run_hybrid_stack.py", data_dir, ds, res, search, folds_dir,
             f"full_dir={full_dir}", f"fold_search={fold_search}", f"models={MODELS}"], cwd=REPO)

## 3. Kiểm tra và run_info.json

In [ ]:
import pandas as pd

crit, diag = {}, {}
for ds in DATASETS:
    res = pd.read_csv(f"{WORK}/results_hybrid_stack_{ds}.csv")
    cnt = res.groupby(["model", "seed"]).size()
    crit[ds] = {"du_3_seed_1_dong": bool(len(res) == 3 and (cnt == 1).all() and set(res["seed"]) == {42, 43, 44}),
                "chi_split_val": set(res["split"]) == {"val"},
                "f1_macro_mean": round(float(res["f1_macro"].mean()), 4), "fpr_mean": round(float(res["fpr"].mean()), 4),
                "size_kb_max": float(res["size_kb"].max()), "p95_ms_max": float(res["infer_ms_per_window"].max())}
    s = json.load(open(f"{WORK}/hybrid_search_stack_{ds}.json", encoding="utf-8"))["hybrid_gat_lgbm_stack"]["seeds"]
    diag[ds] = {k: v["diag"] for k, v in s.items()}
    print(ds, crit[ds], "\n", json.dumps(diag[ds], ensure_ascii=False, indent=1))
INFO["criteria"], INFO["diag"] = crit, diag
INFO["models_hybrid_dirs"] = sorted(os.listdir(MODELS))
save_info()
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())